# EDA, Tuberculosis in the United States


In [ ]:
# Acknowledgments (tutorial, code(modified)): Lisa Yan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive/')

# Tuberculosis in the United States

What can we say about the presence of Tuberculosis in the United States?

Let's look at the data included in the [original CDC article](https://www.cdc.gov/mmwr/volumes/71/wr/mm7112a1.htm?s_cid=mm7112a1_w#T1_down) published in 2021.

<br>

---

# CSV and Nice Field Names
Please use the CSV file `cdc_tuberculosis.csv`.

We can then explore the CSV (which is a text file, and does not contain binary-encoded data) in many ways:
1. Using a the jupyter lab explorer tool look at the data
2. Opening the CSV directly in DataHub (read-only), Excel, Google Sheets, etc.
3. The Python file object
4. pandas, using `pd.read_csv()`

In [4]:
df_tb = pd.read_csv("./cdc_tuberculosis.csv")
df_tb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       53 non-null     object 
 1   No. of TB cases  53 non-null     object 
 2   Unnamed: 2       53 non-null     object 
 3   Unnamed: 3       53 non-null     object 
 4   TB incidence     53 non-null     float64
 5   Unnamed: 5       53 non-null     float64
 6   Unnamed: 6       53 non-null     float64
dtypes: float64(3), object(4)
memory usage: 3.0+ KB


<br>


---

## Play with the data in the Jupyter Lab Explorer
1, 2. Let's start with the first two so we really solidify the idea of a CSV as **rectangular data (i.e., tabular data) stored as comma-separated values**.

<br>

---

## Play with the data in python

3. Next, let's try using the Python file object. Let's check out the first four lines:

In [2]:
#!ls
#!ls "/content/drive/My Drive/Colab Notebooks/data"

# with open("/content/drive/My Drive/Colab Notebooks/data/cdc_tuberculosis.csv", "r") as f:
with open("./cdc_tuberculosis.csv", "r") as f:
    for i, row in enumerate(f):
        print(row)
        if i >= 3: break


,No. of TB cases,,,TB incidence,,

U.S. jurisdiction,2019,2020,2021,2019,2020,2021

Total,"8,900","7,173","7,860",2.71,2.16,2.37

Alabama,87,72,92,1.77,1.43,1.83



Whoa, why are there blank lines interspaced between the lines of the CSV?

In [5]:
with open("./cdc_tuberculosis.csv", "r") as f:
    for i, row in enumerate(f):
        print(repr(row)) # print raw strings
        if i >= 3: break

',No. of TB cases,,,TB incidence,,\n'
'U.S. jurisdiction,2019,2020,2021,2019,2020,2021\n'
'Total,"8,900","7,173","7,860",2.71,2.16,2.37\n'
'Alabama,87,72,92,1.77,1.43,1.83\n'


Wait, what's up with the "Unnamed" column names? And the first row, for that matter?

Congratulations -- you're ready to wrangle your data. Because of how things are stored, we'll need to clean the data a bit to name our columns better.

A reasonable first step is to identify the row with the right header. The `pd.read_csv()` function ([documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)) has the convenient `header` parameter.

You could also try:
1. Shift+Tab while your cursor is in the function call parenthesis.
1. `Cmd+i` or `Ctrl+i` to get contextual help.

In [6]:
tb_df = pd.read_csv("./cdc_tuberculosis.csv", header=1) # row index
tb_df

,U.S. jurisdiction,2019,2020,2021,2019.1,2020.1,2021.1
0,Total,"8,900","7,173","7,860",2.71,2.16,2.37
1,Alabama,87,72,92,1.77,1.43,1.83
2,Alaska,58,58,58,7.91,7.92,7.92
3,Arizona,183,136,129,2.51,1.89,1.77
4,Arkansas,64,59,69,2.12,1.96,2.28
5,California,"2,111","1,706","1,750",5.35,4.32,4.46
6,Colorado,66,52,58,1.15,0.90,1.00
7,Connecticut,67,54,54,1.88,1.50,1.50
8,Delaware,18,17,43,1.84,1.71,4.29
9,District of Columbia,24,19,19,3.39,2.75,2.84


<br><br>
Wait...but now we can't differentiate betwen the "Number of TB cases" and "TB incidence" year columns. pandas has tried to make our lives easier by automatically adding ".1" to the latter columns, but this doesn't help us as humans understand the data.

We can do this manually with `df.rename()` ([documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html?highlight=rename#pandas.DataFrame.rename)):

In [7]:
rename_dict = {'2019': 'TB cases 2019',
               '2020': 'TB cases 2020',
               '2021': 'TB cases 2021',
               '2019.1': 'TB incidence 2019',
               '2020.1': 'TB incidence 2020',
               '2021.1': 'TB incidence 2021'}
tb_df = tb_df.rename(columns=rename_dict)
tb_df

,U.S. jurisdiction,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021
0,Total,"8,900","7,173","7,860",2.71,2.16,2.37
1,Alabama,87,72,92,1.77,1.43,1.83
2,Alaska,58,58,58,7.91,7.92,7.92
3,Arizona,183,136,129,2.51,1.89,1.77
4,Arkansas,64,59,69,2.12,1.96,2.28
5,California,"2,111","1,706","1,750",5.35,4.32,4.46
6,Colorado,66,52,58,1.15,0.90,1.00
7,Connecticut,67,54,54,1.88,1.50,1.50
8,Delaware,18,17,43,1.84,1.71,4.29
9,District of Columbia,24,19,19,3.39,2.75,2.84


<br/><br/>

---

# Record Granularity

You might already be wondering: What's up with that first record?

Row 0 is what we call a **rollup record**, or summary record. It's often useful when displaying tables to humans. The **granularity** of record 0 (Totals) vs the rest of the records (States) is different.

In [8]:
tb_df.head()

,U.S. jurisdiction,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021
0,Total,"8,900","7,173","7,860",2.71,2.16,2.37
1,Alabama,87,72,92,1.77,1.43,1.83
2,Alaska,58,58,58,7.91,7.92,7.92
3,Arizona,183,136,129,2.51,1.89,1.77
4,Arkansas,64,59,69,2.12,1.96,2.28


<br/>

Okay, EDA step two. How was the rollup record aggregated?

Let's check if Total TB cases is the sum of all state TB cases. We can drop it and try to sum up all the remaining rows.

In [9]:
tb_df.drop(0)

,U.S. jurisdiction,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021
1,Alabama,87,72,92,1.77,1.43,1.83
2,Alaska,58,58,58,7.91,7.92,7.92
3,Arizona,183,136,129,2.51,1.89,1.77
4,Arkansas,64,59,69,2.12,1.96,2.28
5,California,"2,111","1,706","1,750",5.35,4.32,4.46
6,Colorado,66,52,58,1.15,0.90,1.00
7,Connecticut,67,54,54,1.88,1.50,1.50
8,Delaware,18,17,43,1.84,1.71,4.29
9,District of Columbia,24,19,19,3.39,2.75,2.84
10,Florida,558,412,499,2.60,1.91,2.29


In [ ]:
tb_df.drop(0).sum()

<br/>

Whoa, what's going on? Check out the column types:

In [10]:
tb_df.dtypes

U.S. jurisdiction     object
TB cases 2019         object
TB cases 2020         object
TB cases 2021         object
TB incidence 2019    float64
TB incidence 2020    float64
TB incidence 2021    float64
dtype: object

<br/>

Looks like those commas are causing all TB cases to be read as the `object` datatype, or **storage type** (close to the Python string datatype), so pandas is concatenating strings instead of adding integers.

<br/>

Fortunately `read_csv` also has a `thousands` parameter (for what it's worth, I didn't know this beforehand--I [googled](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) this):

In [12]:
# improve readability: chaining method calls with outer parentheses/line breaks
tb_df = (
    pd.read_csv("./cdc_tuberculosis.csv", header=1, thousands=',')
    .rename(columns=rename_dict)
)
tb_df

,U.S. jurisdiction,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021
0,Total,8900,7173,7860,2.71,2.16,2.37
1,Alabama,87,72,92,1.77,1.43,1.83
2,Alaska,58,58,58,7.91,7.92,7.92
3,Arizona,183,136,129,2.51,1.89,1.77
4,Arkansas,64,59,69,2.12,1.96,2.28
5,California,2111,1706,1750,5.35,4.32,4.46
6,Colorado,66,52,58,1.15,0.90,1.00
7,Connecticut,67,54,54,1.88,1.50,1.50
8,Delaware,18,17,43,1.84,1.71,4.29
9,District of Columbia,24,19,19,3.39,2.75,2.84


In [ ]:
tb_df.drop(0).sum()

In [13]:
tb_df.head(1)

,U.S. jurisdiction,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021
0,Total,8900,7173,7860,2.71,2.16,2.37


The Total TB cases look right. Phew!

(We'll leave it to your own EDA to figure out how the TB incidence "Totals" were aggregated.)

In order to compute incidence we are going to need more data: **population information**!!

<br/><br/>

---

# Gather Census Data

U.S. Census population estimates [source](https://www.census.gov/data/tables/time-series/demo/popest/2010s-state-total.html) (2019), [source](https://www.census.gov/data/tables/time-series/demo/popest/2020s-state-total.html) (2020-2021).

Running the below cells cleans the data. We encourage you to closely explore the CSV and study these lines after lecture...

There are a few new methods here:
* `df.convert_dtypes()` ([documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.convert_dtypes.html)) conveniently converts all float dtypes into ints and is out of scope for the class.
* `df.drop_na()` ([documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html)) will be explained in more detail next time.

In [14]:
census_2010s_df = pd.read_csv("./nst-est2019-01.csv", header=3, thousands=",")
census_2010s_df

,Unnamed: 0,Census,Estimates Base,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,United States,308745538.0,308758105.0,309321666.0,311556874.0,313830990.0,315993715.0,318301008.0,320635163.0,322941311.0,324985539.0,326687501.0,328239523.0
1,Northeast,55317240.0,55318443.0,55380134.0,55604223.0,55775216.0,55901806.0,56006011.0,56034684.0,56042330.0,56059240.0,56046620.0,55982803.0
2,Midwest,66927001.0,66929725.0,66974416.0,67157800.0,67336743.0,67560379.0,67745167.0,67860583.0,67987540.0,68126781.0,68236628.0,68329004.0
3,South,114555744.0,114563030.0,114866680.0,116006522.0,117241208.0,118364400.0,119624037.0,120997341.0,122351760.0,123542189.0,124569433.0,125580448.0
4,West,71945553.0,71946907.0,72100436.0,72788329.0,73477823.0,74167130.0,74925793.0,75742555.0,76559681.0,77257329.0,77834820.0,78347268.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,Note: The estimates are based on the 2010 Cens...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59,Suggested Citation:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60,Table 1. Annual Estimates of the Resident Popu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61,"Source: U.S. Census Bureau, Population Division",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Do some basic data cleaning

In [15]:
census_2010s_df = (
    census_2010s_df
    .rename(columns={"Unnamed: 0": "Geographic Area"})
    .drop(columns=["Census", "Estimates Base"])
    .convert_dtypes() # "smart" converting of columns to int, use at your own risk
    .dropna()  # we'll introduce this very soon
)
census_2010s_df

,Geographic Area,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,United States,309321666,311556874,313830990,315993715,318301008,320635163,322941311,324985539,326687501,328239523
1,Northeast,55380134,55604223,55775216,55901806,56006011,56034684,56042330,56059240,56046620,55982803
2,Midwest,66974416,67157800,67336743,67560379,67745167,67860583,67987540,68126781,68236628,68329004
3,South,114866680,116006522,117241208,118364400,119624037,120997341,122351760,123542189,124569433,125580448
4,West,72100436,72788329,73477823,74167130,74925793,75742555,76559681,77257329,77834820,78347268
5,.Alabama,4785437,4799069,4815588,4830081,4841799,4852347,4863525,4874486,4887681,4903185
6,.Alaska,713910,722128,730443,737068,736283,737498,741456,739700,735139,731545
7,.Arizona,6407172,6472643,6554978,6632764,6730413,6829676,6941072,7044008,7158024,7278717
8,.Arkansas,2921964,2940667,2952164,2959400,2967392,2978048,2989918,3001345,3009733,3017804
9,.California,37319502,37638369,37948800,38260787,38596972,38918045,39167117,39358497,39461588,39512223


What is the granularity of each row in this table.

Notice there is a `'.'` at the beginning of all the states.  We need to remove that.

In [16]:
census_2010s_df['Geographic Area'] = census_2010s_df['Geographic Area'].str.strip('.')
census_2010s_df

,Geographic Area,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,United States,309321666,311556874,313830990,315993715,318301008,320635163,322941311,324985539,326687501,328239523
1,Northeast,55380134,55604223,55775216,55901806,56006011,56034684,56042330,56059240,56046620,55982803
2,Midwest,66974416,67157800,67336743,67560379,67745167,67860583,67987540,68126781,68236628,68329004
3,South,114866680,116006522,117241208,118364400,119624037,120997341,122351760,123542189,124569433,125580448
4,West,72100436,72788329,73477823,74167130,74925793,75742555,76559681,77257329,77834820,78347268
5,Alabama,4785437,4799069,4815588,4830081,4841799,4852347,4863525,4874486,4887681,4903185
6,Alaska,713910,722128,730443,737068,736283,737498,741456,739700,735139,731545
7,Arizona,6407172,6472643,6554978,6632764,6730413,6829676,6941072,7044008,7158024,7278717
8,Arkansas,2921964,2940667,2952164,2959400,2967392,2978048,2989918,3001345,3009733,3017804
9,California,37319502,37638369,37948800,38260787,38596972,38918045,39167117,39358497,39461588,39512223


## Loading the 2020s data
The 2020s data is in a separate file so we will repeate the same data cleaning process.

In [17]:
# census 2020s data
census_2020s_df = pd.read_csv("./NST-EST2022-POP.csv", header=3, thousands=",")
census_2020s_df = (
    census_2020s_df
    .drop(columns=["Unnamed: 1"])
    .rename(columns={"Unnamed: 0": "Geographic Area"})
    .convert_dtypes()
    .dropna()
)
census_2020s_df['Geographic Area'] = census_2020s_df['Geographic Area'].str.strip('.')
census_2020s_df

,Geographic Area,2020,2021,2022
0,United States,331511512,332031554,333287557
1,Northeast,57448898,57259257,57040406
2,Midwest,68961043,68836505,68787595
3,South,126450613,127346029,128716192
4,West,78650958,78589763,78743364
5,Alabama,5031362,5049846,5074296
6,Alaska,732923,734182,733583
7,Arizona,7179943,7264877,7359197
8,Arkansas,3014195,3028122,3045637
9,California,39501653,39142991,39029342


<br/><br/>

---

# Join Data (Merge DataFrames)

Time to `merge`! Here I use the DataFrame method `df1.merge(right=df2, ...)` on DataFrame `df1` ([documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)). Contrast this with the function `pd.merge(left=df1, right=df2, ...)` ([documentation](https://pandas.pydata.org/docs/reference/api/pandas.merge.html?highlight=pandas%20merge#pandas.merge)). Feel free to use either.

In [18]:
# Show the three tables that we are going to join
display(tb_df.tail(2))
display(census_2010s_df.tail(2))
display(census_2020s_df.tail(2))

,U.S. jurisdiction,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021
50,Wisconsin,51,35,66,0.88,0.59,1.12
51,Wyoming,1,0,3,0.17,0.00,0.52


,Geographic Area,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
55,Wyoming,564487,567299,576305,582122,582531,585613,584215,578931,577601,578759
57,Puerto Rico,3721525,3678732,3634488,3593077,3534874,3473232,3406672,3325286,3193354,3193694


,Geographic Area,2020,2021,2022
55,Wyoming,577605,579483,581381
57,Puerto Rico,3281557,3262693,3221789


In [19]:
# merge TB dataframe with two US census dataframes
tb_census_df = (
    tb_df
    .merge(right=census_2010s_df,
           left_on="U.S. jurisdiction", right_on="Geographic Area")
    .merge(right=census_2020s_df,
           left_on="U.S. jurisdiction", right_on="Geographic Area")
)
tb_census_df.tail()

,U.S. jurisdiction,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021,Geographic Area_x,2010,2011,...,2014,2015,2016,2017,2018,2019,Geographic Area_y,2020,2021,2022
46,Virginia,191,169,161,2.23,1.96,1.86,Virginia,8023699,8101155,...,8310993,8361808,8410106,8463587,8501286,8535519,Virginia,8636471,8657365,8683619
47,Washington,221,163,199,2.90,2.11,2.57,Washington,6742830,6826627,...,7054655,7163657,7294771,7423362,7523869,7614893,Washington,7724031,7740745,7785786
48,West Virginia,9,13,7,0.50,0.73,0.39,West Virginia,1854239,1856301,...,1849489,1842050,1831023,1817004,1804291,1792147,West Virginia,1791420,1785526,1775156
49,Wisconsin,51,35,66,0.88,0.59,1.12,Wisconsin,5690475,5705288,...,5751525,5760940,5772628,5790186,5807406,5822434,Wisconsin,5896271,5880101,5892539
50,Wyoming,1,0,3,0.17,0.00,0.52,Wyoming,564487,567299,...,582531,585613,584215,578931,577601,578759,Wyoming,577605,579483,581381


This is a little unwieldy. We could either drop the unneeded columns now, or just merge on smaller census DataFrames. Let's do the latter.

In [20]:
# try merging again, but cleaner this time
tb_census_df = (
    tb_df
    .merge(right=census_2010s_df[["Geographic Area", "2019"]],
           left_on="U.S. jurisdiction", right_on="Geographic Area")
    .drop(columns="Geographic Area")
    .merge(right=census_2020s_df[["Geographic Area", "2020", "2021"]],
           left_on="U.S. jurisdiction", right_on="Geographic Area")
    .drop(columns="Geographic Area")
)
tb_census_df.tail()

,U.S. jurisdiction,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021,2019,2020,2021
46,Virginia,191,169,161,2.23,1.96,1.86,8535519,8636471,8657365
47,Washington,221,163,199,2.90,2.11,2.57,7614893,7724031,7740745
48,West Virginia,9,13,7,0.50,0.73,0.39,1792147,1791420,1785526
49,Wisconsin,51,35,66,0.88,0.59,1.12,5822434,5896271,5880101
50,Wyoming,1,0,3,0.17,0.00,0.52,578759,577605,579483


## Reproduce incidence

Let's recompute incidence to make sure we know where the original CDC numbers came from.

From the [CDC report](https://www.cdc.gov/mmwr/volumes/71/wr/mm7112a1.htm?s_cid=mm7112a1_w#T1_down): TB incidence is computed as “Cases per 100,000 persons using mid-year population estimates from the U.S. Census Bureau.”

If we define a group as 100,000 people, then we can compute the TB incidence for a given state population as

$$\text{TB incidence} = \frac{\text{\# TB cases in population}}{\text{\# groups in population}} = \frac{\text{\# TB cases in population}}{\text{population}/100000} $$

$$= \frac{\text{\# TB cases in population}}{\text{population}} \times 100000$$

Let's try this for 2019:

In [21]:
tb_census_df["recompute incidence 2019"] = (
    tb_census_df["TB cases 2019"]/tb_census_df["2019"] * 100_000
)
tb_census_df

,U.S. jurisdiction,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021,2019,2020,2021,recompute incidence 2019
0,Alabama,87,72,92,1.77,1.43,1.83,4903185,5031362,5049846,1.774357
1,Alaska,58,58,58,7.91,7.92,7.92,731545,732923,734182,7.928425
2,Arizona,183,136,129,2.51,1.89,1.77,7278717,7179943,7264877,2.514179
3,Arkansas,64,59,69,2.12,1.96,2.28,3017804,3014195,3028122,2.120747
4,California,2111,1706,1750,5.35,4.32,4.46,39512223,39501653,39142991,5.342651
5,Colorado,66,52,58,1.15,0.90,1.00,5758736,5784865,5811297,1.146085
6,Connecticut,67,54,54,1.88,1.50,1.50,3565287,3597362,3623355,1.879232
7,Delaware,18,17,43,1.84,1.71,4.29,973764,992114,1004807,1.848497
8,District of Columbia,24,19,19,3.39,2.75,2.84,705749,670868,668791,3.400642
9,Florida,558,412,499,2.60,1.91,2.29,21477737,21589602,21828069,2.598039


Awesome!!!

Let's use a for-loop and Python format strings to compute TB incidence for all years. Python f-strings are just used for the purposes of this demo, but they're handy to know when you explore data beyond this course ([Python documentation](https://docs.python.org/3/tutorial/inputoutput.html)).

In [22]:
# recompute incidence for all years
for year in [2019, 2020, 2021]:
    tb_census_df[f"recompute incidence {year}"] = (
        tb_census_df[f"TB cases {year}"]/tb_census_df[f"{year}"]*100_000
    )
tb_census_df

,U.S. jurisdiction,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021,2019,2020,2021,recompute incidence 2019,recompute incidence 2020,recompute incidence 2021
0,Alabama,87,72,92,1.77,1.43,1.83,4903185,5031362,5049846,1.774357,1.431024,1.821838
1,Alaska,58,58,58,7.91,7.92,7.92,731545,732923,734182,7.928425,7.913519,7.899949
2,Arizona,183,136,129,2.51,1.89,1.77,7278717,7179943,7264877,2.514179,1.894165,1.775667
3,Arkansas,64,59,69,2.12,1.96,2.28,3017804,3014195,3028122,2.120747,1.957405,2.27864
4,California,2111,1706,1750,5.35,4.32,4.46,39512223,39501653,39142991,5.342651,4.318807,4.470788
5,Colorado,66,52,58,1.15,0.90,1.00,5758736,5784865,5811297,1.146085,0.898897,0.998056
6,Connecticut,67,54,54,1.88,1.50,1.50,3565287,3597362,3623355,1.879232,1.5011,1.490331
7,Delaware,18,17,43,1.84,1.71,4.29,973764,992114,1004807,1.848497,1.713513,4.279429
8,District of Columbia,24,19,19,3.39,2.75,2.84,705749,670868,668791,3.400642,2.832152,2.840947
9,Florida,558,412,499,2.60,1.91,2.29,21477737,21589602,21828069,2.598039,1.908326,2.286047


These numbers look pretty close!!! There are a few errors in the hundredths place, particularly in 2021. It may be useful to further explore reasons behind this discrepancy. We'll leave it to you!

In [23]:
tb_census_df.describe()

,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021,2019,2020,2021,recompute incidence 2019,recompute incidence 2020,recompute incidence 2021
count,51.000000,51.000000,51.000000,51.000000,51.000000,51.000000,51.0,51.0,51.0,51.0,51.0,51.0
mean,174.509804,140.647059,154.117647,2.102549,1.782941,1.971961,6436069.078431,6500225.72549,6510422.627451,2.104969,1.784655,1.969928
std,341.738752,271.055775,286.781007,1.498745,1.337414,1.478468,7360660.467814,7408168.462614,7394300.076705,1.500236,1.338263,1.474929
min,1.000000,0.000000,2.000000,0.170000,0.000000,0.210000,578759.0,577605.0,579483.0,0.172783,0.0,0.210049
25%,25.500000,29.000000,23.000000,1.295000,1.210000,1.235000,1789606.0,1820311.0,1844920.0,1.297485,1.211433,1.233905
50%,70.000000,67.000000,69.000000,1.800000,1.520000,1.700000,4467673.0,4507445.0,4506589.0,1.808606,1.521612,1.694502
75%,180.500000,139.000000,150.000000,2.575000,1.990000,2.220000,7446805.0,7451987.0,7502811.0,2.577577,1.993607,2.219482
max,2111.000000,1706.000000,1750.000000,7.910000,7.920000,7.920000,39512223.0,39501653.0,39142991.0,7.928425,7.913519,7.899949



<br><br><br>

---

# Bonus EDA

**How do we reproduce that reported statistic in the original [CDC report](https://www.cdc.gov/mmwr/volumes/71/wr/mm7112a1.htm?s_cid=mm7112a1_w)?**

> Reported TB incidence (cases per 100,000 persons) increased **9.4%**, from **2.2** during 2020 to **2.4** during 2021 but was lower than incidence during 2019 (2.7). Increases occurred among both U.S.-born and non–U.S.-born persons.

This is TB incidence computed across the entire U.S. population! How do we reproduce this
* We need to reproduce the "Total" TB incidences in our rolled record.
* But our current `tb_census_df` only has 51 entries (50 states plus Washington, D.C.). There is no rolled record.
* What happened...?

Let's get exploring!
<br/><br/>

Before we keep exploring, I'm going to set all indexes to more meaningful values, instead of just numbers that pertained to some row at some point. This will make our cleaning slightly easier.

In [24]:
tb_df = tb_df.set_index("U.S. jurisdiction")
tb_df

,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021
U.S. jurisdiction,,,,,,
Total,8900,7173,7860,2.71,2.16,2.37
Alabama,87,72,92,1.77,1.43,1.83
Alaska,58,58,58,7.91,7.92,7.92
Arizona,183,136,129,2.51,1.89,1.77
Arkansas,64,59,69,2.12,1.96,2.28
California,2111,1706,1750,5.35,4.32,4.46
Colorado,66,52,58,1.15,0.90,1.00
Connecticut,67,54,54,1.88,1.50,1.50
Delaware,18,17,43,1.84,1.71,4.29


In [ ]:
census_2010s_df = census_2010s_df.set_index("Geographic Area")
census_2010s_df

In [25]:
census_2020s_df = census_2020s_df.set_index("Geographic Area")
census_2020s_df

,2020,2021,2022
Geographic Area,,,
United States,331511512,332031554,333287557
Northeast,57448898,57259257,57040406
Midwest,68961043,68836505,68787595
South,126450613,127346029,128716192
West,78650958,78589763,78743364
Alabama,5031362,5049846,5074296
Alaska,732923,734182,733583
Arizona,7179943,7264877,7359197
Arkansas,3014195,3028122,3045637


It turns out that our merge above only kept state records, even though our original `tb_df` had the "Total" rolled record:

In [26]:
tb_df.head()

,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021
U.S. jurisdiction,,,,,,
Total,8900,7173,7860,2.71,2.16,2.37
Alabama,87,72,92,1.77,1.43,1.83
Alaska,58,58,58,7.91,7.92,7.92
Arizona,183,136,129,2.51,1.89,1.77
Arkansas,64,59,69,2.12,1.96,2.28


Recall that merge by default does an **inner** merge by default, meaning that it only preserves keys that are present in **both** DataFrames.

The rolled records in our census dataframes have different `Geographic Area` fields, which was the key we merged on:

In [27]:
census_2010s_df

,Geographic Area,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,United States,309321666,311556874,313830990,315993715,318301008,320635163,322941311,324985539,326687501,328239523
1,Northeast,55380134,55604223,55775216,55901806,56006011,56034684,56042330,56059240,56046620,55982803
2,Midwest,66974416,67157800,67336743,67560379,67745167,67860583,67987540,68126781,68236628,68329004
3,South,114866680,116006522,117241208,118364400,119624037,120997341,122351760,123542189,124569433,125580448
4,West,72100436,72788329,73477823,74167130,74925793,75742555,76559681,77257329,77834820,78347268
5,Alabama,4785437,4799069,4815588,4830081,4841799,4852347,4863525,4874486,4887681,4903185
6,Alaska,713910,722128,730443,737068,736283,737498,741456,739700,735139,731545
7,Arizona,6407172,6472643,6554978,6632764,6730413,6829676,6941072,7044008,7158024,7278717
8,Arkansas,2921964,2940667,2952164,2959400,2967392,2978048,2989918,3001345,3009733,3017804
9,California,37319502,37638369,37948800,38260787,38596972,38918045,39167117,39358497,39461588,39512223


The Census DataFrame has several rolled records. The aggregate record we are looking for actually has the Geographic Area named "United States".

One straightforward way to get the right merge is to rename the value itself. Because we now have the Geographic Area index, we'll use `df.rename()` ([documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html)):

In [28]:
# rename rolled record for 2010s
census_2010s_df.rename(index={'United States':'Total'}, inplace=True)
census_2010s_df

,Geographic Area,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,United States,309321666,311556874,313830990,315993715,318301008,320635163,322941311,324985539,326687501,328239523
1,Northeast,55380134,55604223,55775216,55901806,56006011,56034684,56042330,56059240,56046620,55982803
2,Midwest,66974416,67157800,67336743,67560379,67745167,67860583,67987540,68126781,68236628,68329004
3,South,114866680,116006522,117241208,118364400,119624037,120997341,122351760,123542189,124569433,125580448
4,West,72100436,72788329,73477823,74167130,74925793,75742555,76559681,77257329,77834820,78347268
5,Alabama,4785437,4799069,4815588,4830081,4841799,4852347,4863525,4874486,4887681,4903185
6,Alaska,713910,722128,730443,737068,736283,737498,741456,739700,735139,731545
7,Arizona,6407172,6472643,6554978,6632764,6730413,6829676,6941072,7044008,7158024,7278717
8,Arkansas,2921964,2940667,2952164,2959400,2967392,2978048,2989918,3001345,3009733,3017804
9,California,37319502,37638369,37948800,38260787,38596972,38918045,39167117,39358497,39461588,39512223


In [ ]:
# same, but for 2020s rename rolled record
census_2020s_df.rename(index={'United States':'Total'}, inplace=True)
census_2020s_df

<br/>

Next let's rerun our merge. Note the different chaining, because we are now merging on indexes (`df.merge()` [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)).

In [29]:
tb_census_df = (
    tb_df
    .merge(right=census_2010s_df[["2019"]],
           left_index=True, right_index=True)
    .merge(right=census_2020s_df[["2020", "2021"]],
           left_index=True, right_index=True)
)
tb_census_df

,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021,2019,2020,2021


<br/>

Finally, let's recompute our incidences:

In [39]:
# recompute incidence for all years
for year in [2019, 2020, 2021]:
    tb_census_df[f"recompute incidence {year}"] = tb_census_df[f"TB cases {year}"]/tb_census_df[f"{year}"]*100000
tb_census_df

,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021,2019,2020,2021,recompute incidence 2019,recompute incidence 2020,recompute incidence 2021


We reproduced the total U.S. incidences correctly!

We're almost there. Let's revisit the quote:

> Reported TB incidence (cases per 100,000 persons) increased **9.4%**, from **2.2** during 2020 to **2.4** during 2021 but was lower than incidence during 2019 (2.7). Increases occurred among both U.S.-born and non–U.S.-born persons.

Recall that percent change from $A$ to $B$ is computed as
$$\text{percent change} = \frac{B - A}{A} \times 100$$.

In [35]:
tb_census_df

,TB cases 2019,TB cases 2020,TB cases 2021,TB incidence 2019,TB incidence 2020,TB incidence 2021,2019,2020,2021,recompute incidence 2019,recompute incidence 2020,recompute incidence 2021


In [43]:
# incidence_2020 = tb_census_df.loc[:, ['Total', 'recompute incidence 2020']]
incidence_2020 = tb_census_df.iloc[:, [7, 10]]
incidence_2020

,2020,recompute incidence 2020


In [44]:
# incidence_2021 = tb_census_df.loc['Total', 'recompute incidence 2021']
incidence_2021 = tb_census_df.iloc[:, [8, 11]]
incidence_2021


,2021,recompute incidence 2021


In [45]:
difference = (incidence_2021 - incidence_2020)/incidence_2020 * 100
difference

,2020,2021,recompute incidence 2020,recompute incidence 2021


We did it!!!